# Gates Masculinity Project — OpenAI API LLM EDA Notebook

This notebook runs a full OpenAI API-assisted coding and exploratory analysis pipeline for:

1. **Audience comments** — `Kenya Audience Analysis Comments.xlsx`
2. **Content snippets** — `Kenya Content Analysis Snippets.xlsx`

It produces cleaned data, coded data, summary tables, representative quotes, flagged rows for human review, and a written Markdown report.

Run the notebook top to bottom. You only need to provide the two Excel files and your OpenAI API key.

In [ ]:
# Optional: install dependencies if missing
import sys, subprocess, importlib.util

def ensure_package(pkg, import_name=None):
    import_name = import_name or pkg
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

for pkg, import_name in [
    ("pandas", "pandas"),
    ("openpyxl", "openpyxl"),
    ("openai", "openai"),
    ("tqdm", "tqdm"),
    ("matplotlib", "matplotlib"),
    ("tabulate", "tabulate"),
]:
    ensure_package(pkg, import_name)

print("Dependencies ready.")

In [ ]:
import os, re, json, time, math, textwrap
from pathlib import Path
from datetime import datetime
from getpass import getpass

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from openai import OpenAI

# -----------------------------
# USER CONFIGURATION
# -----------------------------

# Put these Excel files in the same folder as this notebook, or update paths below.
AUDIENCE_FILE = Path("Kenya Audience Analysis Comments.xlsx")
CONTENT_FILE = Path("Kenya Content Analysis Snippets.xlsx")

OUTPUT_DIR = Path("outputs_llm_eda")
OUTPUT_DIR.mkdir(exist_ok=True)

# Choose model. Good defaults: "gpt-4.1-mini" for cheaper coding, "gpt-4.1" for stronger coding.
# If your account has another preferred model, replace this.
MODEL = "gpt-4.1-mini"

# Batch size: 10 is safer; 20 is faster.
BATCH_SIZE = 10

# For testing, set to small numbers like 20. For full run, keep None.
MAX_AUDIENCE_ROWS = None
MAX_CONTENT_ROWS = None

# If interrupted, rerun notebook and this will resume from checkpoints.
RESUME_FROM_CHECKPOINTS = True
TEMPERATURE = 0

print(f"Output folder: {OUTPUT_DIR.resolve()}")

In [ ]:
# Enter API key securely. It will not be printed.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("OpenAI client ready.")

## 1. Load and inspect datasets

In [ ]:
def log(msg):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {msg}"
    print(line)
    with open(OUTPUT_DIR / "analysis_log.txt", "a", encoding="utf-8") as f:
        f.write(line + "\n")

with open(OUTPUT_DIR / "analysis_log.txt", "w", encoding="utf-8") as f:
    f.write(f"Analysis started: {datetime.now().isoformat()}\n")

if not AUDIENCE_FILE.exists():
    raise FileNotFoundError(f"Cannot find audience file: {AUDIENCE_FILE.resolve()}")
if not CONTENT_FILE.exists():
    raise FileNotFoundError(f"Cannot find content file: {CONTENT_FILE.resolve()}")

aud_raw = pd.read_excel(AUDIENCE_FILE)
cont_raw = pd.read_excel(CONTENT_FILE)

if MAX_AUDIENCE_ROWS:
    aud_raw = aud_raw.head(MAX_AUDIENCE_ROWS).copy()
if MAX_CONTENT_ROWS:
    cont_raw = cont_raw.head(MAX_CONTENT_ROWS).copy()

log(f"Loaded audience file: {AUDIENCE_FILE}, shape={aud_raw.shape}")
log(f"Loaded content file: {CONTENT_FILE}, shape={cont_raw.shape}")

print("Audience columns:")
print(list(aud_raw.columns))
print("\nContent columns:")
print(list(cont_raw.columns))

display(aud_raw.head(3))
display(cont_raw.head(3))

## 2. Auto-detect key columns

Check the printed choices. If wrong, set an override variable to the exact column name and rerun this cell.

In [ ]:
# Optional manual overrides. Leave as None unless auto-detection is wrong.
AUDIENCE_COMMENT_COL_OVERRIDE = None
AUDIENCE_POST_COL_OVERRIDE = None
AUDIENCE_INFLUENCER_COL_OVERRIDE = None
AUDIENCE_PLATFORM_COL_OVERRIDE = None
AUDIENCE_COUNTRY_COL_OVERRIDE = None

CONTENT_SNIPPET_COL_OVERRIDE = None
CONTENT_INFLUENCER_COL_OVERRIDE = None
CONTENT_PLATFORM_COL_OVERRIDE = None
CONTENT_COUNTRY_COL_OVERRIDE = None


def normalize_col_name(c):
    return re.sub(r"[^a-z0-9]+", "_", str(c).lower()).strip("_")


def score_text_column(series):
    s = series.dropna().astype(str)
    if len(s) == 0:
        return -1
    avg_len = s.map(len).mean()
    nonempty = (s.str.strip().str.len() > 0).mean()
    unique_ratio = s.nunique() / max(len(s), 1)
    return avg_len * 0.65 + nonempty * 50 + unique_ratio * 25


def find_col(df, keywords, override=None, prefer_long_text=False):
    if override and override in df.columns:
        return override
    normalized = {c: normalize_col_name(c) for c in df.columns}
    best, best_score = None, -1
    for c, nc in normalized.items():
        kw_score = sum(1 for kw in keywords if kw in nc)
        if kw_score > 0:
            text_score = score_text_column(df[c]) if prefer_long_text else 0
            score = kw_score * 100 + text_score
            if score > best_score:
                best, best_score = c, score
    if best is not None:
        return best
    if prefer_long_text:
        scores = {c: score_text_column(df[c]) for c in df.columns}
        return max(scores, key=scores.get)
    return None


aud_cols = {
    "comment": find_col(aud_raw, ["comment", "reply", "text", "body"], AUDIENCE_COMMENT_COL_OVERRIDE, True),
    "post": find_col(aud_raw, ["post", "original", "content", "caption", "tweet", "video", "snippet"], AUDIENCE_POST_COL_OVERRIDE, True),
    "influencer": find_col(aud_raw, ["influencer", "creator", "author", "account", "name"], AUDIENCE_INFLUENCER_COL_OVERRIDE),
    "platform": find_col(aud_raw, ["platform", "source", "site"], AUDIENCE_PLATFORM_COL_OVERRIDE),
    "country": find_col(aud_raw, ["country"], AUDIENCE_COUNTRY_COL_OVERRIDE),
}
if aud_cols["post"] == aud_cols["comment"]:
    aud_cols["post"] = None

cont_cols = {
    "snippet": find_col(cont_raw, ["snippet", "content", "text", "transcript", "post", "quote", "caption"], CONTENT_SNIPPET_COL_OVERRIDE, True),
    "influencer": find_col(cont_raw, ["influencer", "creator", "author", "account", "name"], CONTENT_INFLUENCER_COL_OVERRIDE),
    "platform": find_col(cont_raw, ["platform", "source", "site"], CONTENT_PLATFORM_COL_OVERRIDE),
    "country": find_col(cont_raw, ["country"], CONTENT_COUNTRY_COL_OVERRIDE),
}

log(f"Audience detected columns: {aud_cols}")
log(f"Content detected columns: {cont_cols}")
print("Audience detected columns:", aud_cols)
print("Content detected columns:", cont_cols)

## 3. Clean text and prepare datasets

In [ ]:
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).replace("\u00a0", " ")
    return re.sub(r"\s+", " ", x).strip()


def add_basic_fields(df, text_col, prefix, cols_map):
    out = df.copy()
    out[f"{prefix}_row_id"] = [f"{prefix.upper()}_{i+1:04d}" for i in range(len(out))]
    out["text_original_for_coding"] = out[text_col].astype(str) if text_col else ""
    out["text_clean_for_coding"] = out["text_original_for_coding"].map(clean_text)
    out["word_count"] = out["text_clean_for_coding"].map(lambda x: len(x.split()) if x else 0)
    out["char_count"] = out["text_clean_for_coding"].map(len)
    out["is_empty_or_near_empty"] = out["word_count"] < 2
    out["is_duplicate_text"] = out["text_clean_for_coding"].duplicated(keep=False)
    for key in ["influencer", "platform", "country"]:
        col = cols_map.get(key)
        out[f"std_{key}"] = out[col].astype(str).map(clean_text) if col and col in out.columns else "Unknown"
    return out

if not aud_cols["comment"]:
    raise ValueError("Could not detect audience comment column. Set AUDIENCE_COMMENT_COL_OVERRIDE.")
if not cont_cols["snippet"]:
    raise ValueError("Could not detect content snippet column. Set CONTENT_SNIPPET_COL_OVERRIDE.")

aud = add_basic_fields(aud_raw, aud_cols["comment"], "aud", aud_cols)
cont = add_basic_fields(cont_raw, cont_cols["snippet"], "con", cont_cols)

aud["comment_text_clean"] = aud["text_clean_for_coding"]
aud["original_post_text_clean"] = aud[aud_cols["post"]].map(clean_text) if aud_cols.get("post") and aud_cols["post"] in aud.columns else ""
cont["snippet_text_clean"] = cont["text_clean_for_coding"]

aud.to_csv(OUTPUT_DIR / "audience_cleaned.csv", index=False)
cont.to_csv(OUTPUT_DIR / "content_cleaned.csv", index=False)

log(f"Saved cleaned files. Audience rows={len(aud)}, Content rows={len(cont)}")
display(aud[["aud_row_id", "std_influencer", "std_platform", "comment_text_clean", "word_count"]].head())
display(cont[["con_row_id", "std_influencer", "std_platform", "snippet_text_clean", "word_count"]].head())

## 4. Define codebooks and structured output schemas

In [ ]:
AUDIENCE_CODEBOOK = {
    "relevance": ["relevant", "partially_relevant", "not_relevant"],
    "stance_toward_post": ["supports", "opposes", "mixed_or_qualified", "neutral_unclear", "irrelevant"],
    "sentiment": ["positive", "negative", "mixed", "neutral"],
    "emotion": ["anger", "contempt", "sadness", "fear_anxiety", "hope", "admiration", "humor", "concern", "pride", "neutral_unclear"],
    "audience_frame": ["male_victimhood", "female_blame", "equality_partnership", "provider_pressure", "traditional_gender_order", "trauma_healing", "faith_morality", "self_improvement", "social_accountability", "violence_safety", "unrelated_unclear"],
    "misogyny_sexism": ["none", "mild_stereotype", "benevolent_or_traditional_sexism", "hostile_sexism", "dehumanizing_or_violent"],
    "argument_type": ["no_argument", "personal_experience", "generalization", "moral_religious_claim", "cultural_social_observation", "facts_statistics", "insult_mockery", "advice_instruction", "testimony", "rhetorical_question"],
    "intensifies_or_softens": ["intensifies_original_message", "softens_original_message", "challenges_original_message", "extends_original_message", "unrelated_unclear"],
    "perceived_impact": ["validation", "learning_reflection", "personal_disclosure", "advice_to_others", "resistance_pushback", "emotional_support", "entertainment_humor", "no_clear_impact"],
    "confidence": ["high", "medium", "low"],
    "human_review_flag": ["yes", "no"]
}

CONTENT_CODEBOOK = {
    "relevance": ["relevant", "partially_relevant", "not_relevant"],
    "primary_topic": ["dating_relationships", "marriage_family", "money_status_provision", "fitness_self_improvement", "mental_health", "gender_debate_social_issues", "religion_morality", "violence_safety", "fatherhood_parenting", "sex_sexuality", "other"],
    "masculinity_narrative": ["men_should_dominate_lead", "men_should_provide_succeed", "men_are_disadvantaged_victims", "men_should_improve_themselves", "men_should_be_fully_self_reliant", "men_should_be_emotionally_open", "men_should_not_show_emotions", "men_should_be_equal_partners", "men_should_protect_women_children", "mixed_unclear", "none"],
    "problem_identified": ["women_feminism", "mens_behavior", "economic_status_pressure", "mental_health_emotional_struggle", "western_influence", "society_culture", "violence_abuse", "family_breakdown", "moral_decline", "no_clear_problem", "other"],
    "solution_proposed": ["assert_dominance_control", "improve_wealth_status", "self_discipline_fitness", "emotional_growth_healing", "equality_respect", "faith_morality", "social_political_change", "accountability_consent", "family_responsibility", "community_support", "no_clear_solution", "other"],
    "main_frame": ["male_victimhood", "female_blame", "provider_pressure", "traditional_gender_order", "equality_partnership", "trauma_healing", "faith_morality", "self_improvement", "social_accountability", "violence_safety", "fatherhood_responsibility", "mixed_unclear"],
    "sentiment_toward_men": ["positive", "negative", "mixed", "neutral_unclear"],
    "sentiment_toward_women": ["positive", "negative", "mixed", "neutral_unclear"],
    "sentiment_toward_traditional_gender_norms": ["positive", "negative", "mixed", "neutral_unclear"],
    "rhetorical_style": ["advice_instruction", "personal_story", "commentary_opinion", "debate_argument", "humor_satire", "motivational_speech", "religious_moral_teaching", "news_facts", "warning_threat", "testimony", "other"],
    "argument_type": ["no_argument", "personal_experience", "generalization", "moral_religious_claim", "cultural_social_observation", "facts_statistics", "insult_mockery", "advice_instruction", "testimony", "rhetorical_question"],
    "misogyny_sexism": ["none", "mild_stereotype", "benevolent_or_traditional_sexism", "hostile_sexism", "dehumanizing_or_violent"],
    "content_orientation": ["regressive", "progressive", "mixed", "unclear"],
    "confidence": ["high", "medium", "low"],
    "human_review_flag": ["yes", "no"]
}


def schema_for_batch(codebook, id_field):
    props = {id_field: {"type": "string"}}
    required = [id_field]
    for k, vals in codebook.items():
        props[k] = {"type": "string", "enum": vals}
        required.append(k)
    props["evidence_quote"] = {"type": "string"}
    required.append("evidence_quote")
    props["coding_rationale"] = {"type": "string"}
    required.append("coding_rationale")
    return {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "coded_rows": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": props,
                    "required": required
                }
            }
        },
        "required": ["coded_rows"]
    }

AUDIENCE_SCHEMA = schema_for_batch(AUDIENCE_CODEBOOK, "aud_row_id")
CONTENT_SCHEMA = schema_for_batch(CONTENT_CODEBOOK, "con_row_id")
print("Schemas ready.")

## 5. LLM coding prompts and API helpers

In [ ]:
AUDIENCE_SYSTEM_PROMPT = """
You are a careful computational social science coder for a research project on masculinity and gender norms in Kenya and Nigeria.
You code audience comments using a fixed codebook. Use only allowed labels. Be conservative and auditable.
Stance is toward the original post/message, not toward men or women generally.
Misogyny/sexism should be coded only when the comment expresses gender stereotypes, gender hierarchy, hostility, degradation, coercion, or unequal expectations.
Evidence quote must be a short quote from the comment, ideally under 20 words.
Flag human_review_flag = yes for low confidence, ambiguity, mixed stance, hostile/dehumanizing sexism, sarcasm, slang-heavy rows, or very short but meaningful rows.
Do not overinterpret emoji-only or vague comments. Code them as not_relevant or partially_relevant as appropriate.
""".strip()

CONTENT_SYSTEM_PROMPT = """
You are a careful computational social science coder for a research project on masculinity and gender norms in Kenya and Nigeria.
You code influencer content snippets using a fixed codebook. Use only allowed labels. Be conservative and auditable.
Assess the snippet itself, not the audience reaction.
Evidence quote must be a short quote from the snippet, ideally under 20 words.
Flag human_review_flag = yes for low confidence, mixed/unclear orientation, hostile/dehumanizing sexism, complex multi-claim snippets, possible sarcasm, or multi-speaker ambiguity.
Do not overclaim if the snippet is vague.
""".strip()

AUDIENCE_CODEBOOK_TEXT = json.dumps(AUDIENCE_CODEBOOK, indent=2)
CONTENT_CODEBOOK_TEXT = json.dumps(CONTENT_CODEBOOK, indent=2)


def make_audience_user_prompt(records):
    return f"""
Code the audience comment rows below.

Definitions and allowed labels:
{AUDIENCE_CODEBOOK_TEXT}

Additional definitions:
- relevance relevant = comment discusses the post, gender, masculinity, men, women, relationships, marriage, family, fatherhood, provision, status, faith, trauma, healing, violence, social norms, or moral expectations.
- stance_toward_post = attitude toward the original post/message.
- intensifies_original_message = makes the original message more extreme, hostile, or rigid.
- softens_original_message = makes the original message more moderate, empathetic, nuanced, or less hostile.
- challenges_original_message = disagrees or pushes back.
- extends_original_message = agrees and adds reasoning without making it clearly more extreme.

Rows to code:
{json.dumps(records, ensure_ascii=False, indent=2)}
""".strip()


def make_content_user_prompt(records):
    return f"""
Code the influencer content snippet rows below.

Definitions and allowed labels:
{CONTENT_CODEBOOK_TEXT}

Additional definitions:
- content_orientation regressive = reinforces hierarchy, domination, female submission, misogyny, rigid provider roles, anti-feminism, male grievance.
- content_orientation progressive = promotes equality, accountability, emotional openness, anti-violence, healing, care, partnership.
- content_orientation mixed = contains both or is ambiguous.

Rows to code:
{json.dumps(records, ensure_ascii=False, indent=2)}
""".strip()


def call_openai_structured(system_prompt, user_prompt, schema, schema_name, max_retries=4):
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.responses.create(
                model=MODEL,
                input=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                text={
                    "format": {
                        "type": "json_schema",
                        "name": schema_name,
                        "schema": schema,
                        "strict": True,
                    }
                },
                temperature=TEMPERATURE,
            )
            return json.loads(resp.output_text)
        except Exception as e:
            last_err = e
            wait = 2 ** attempt
            log(f"OpenAI call failed on attempt {attempt+1}/{max_retries}: {e}. Waiting {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"OpenAI call failed after {max_retries} attempts: {last_err}")

print("LLM helpers ready.")

## 6. Batch coding functions with checkpointing

In [ ]:
def batch_iter(df, batch_size):
    for start in range(0, len(df), batch_size):
        yield start, df.iloc[start:start+batch_size].copy()


def prepare_audience_records(batch):
    recs = []
    for _, r in batch.iterrows():
        recs.append({
            "aud_row_id": r["aud_row_id"],
            "influencer": r.get("std_influencer", "Unknown"),
            "platform": r.get("std_platform", "Unknown"),
            "country": r.get("std_country", "Unknown"),
            "original_post_text": r.get("original_post_text_clean", ""),
            "comment_text": r.get("comment_text_clean", ""),
        })
    return recs


def prepare_content_records(batch):
    recs = []
    for _, r in batch.iterrows():
        recs.append({
            "con_row_id": r["con_row_id"],
            "influencer": r.get("std_influencer", "Unknown"),
            "platform": r.get("std_platform", "Unknown"),
            "country": r.get("std_country", "Unknown"),
            "snippet_text": r.get("snippet_text_clean", ""),
        })
    return recs


def code_dataset(df, kind):
    assert kind in ["audience", "content"]
    checkpoint_path = OUTPUT_DIR / f"{kind}_coded_checkpoint.csv"
    id_col = "aud_row_id" if kind == "audience" else "con_row_id"
    coded_existing = pd.DataFrame()
    done_ids = set()
    if RESUME_FROM_CHECKPOINTS and checkpoint_path.exists():
        coded_existing = pd.read_csv(checkpoint_path)
        done_ids = set(coded_existing[id_col].astype(str))
        log(f"Resuming {kind}: found {len(done_ids)} already coded rows")
    rows_to_code = df[~df[id_col].astype(str).isin(done_ids)].copy()
    log(f"Coding {kind}: {len(rows_to_code)} rows remaining")
    all_new = []
    for start, batch in tqdm(list(batch_iter(rows_to_code, BATCH_SIZE)), desc=f"Coding {kind}"):
        if kind == "audience":
            records = prepare_audience_records(batch)
            data = call_openai_structured(AUDIENCE_SYSTEM_PROMPT, make_audience_user_prompt(records), AUDIENCE_SCHEMA, "audience_batch_coding")
        else:
            records = prepare_content_records(batch)
            data = call_openai_structured(CONTENT_SYSTEM_PROMPT, make_content_user_prompt(records), CONTENT_SCHEMA, "content_batch_coding")
        coded_rows = data.get("coded_rows", [])
        if len(coded_rows) != len(batch):
            log(f"Warning: expected {len(batch)} coded rows, got {len(coded_rows)} for {kind} batch starting {start}")
        all_new.extend(coded_rows)
        combined = pd.concat([coded_existing, pd.DataFrame(all_new)], ignore_index=True)
        combined.to_csv(checkpoint_path, index=False)
        time.sleep(0.25)
    final_coded = pd.concat([coded_existing, pd.DataFrame(all_new)], ignore_index=True)
    final_coded = final_coded.drop_duplicates(subset=[id_col], keep="last")
    return final_coded

print("Batch coding ready.")

## 7. Run LLM coding for audience comments

This may take time depending on rows, model, and rate limits.

In [ ]:
aud_coded_only = code_dataset(aud, "audience")
log(f"Audience coding complete: {aud_coded_only.shape}")
display(aud_coded_only.head())

## 8. Run LLM coding for content snippets

In [ ]:
cont_coded_only = code_dataset(cont, "content")
log(f"Content coding complete: {cont_coded_only.shape}")
display(cont_coded_only.head())

## 9. Merge coded labels back into full datasets and save

In [ ]:
aud_full = aud.merge(aud_coded_only, on="aud_row_id", how="left")
cont_full = cont.merge(cont_coded_only, on="con_row_id", how="left")

aud_full.loc[aud_full["is_empty_or_near_empty"], "human_review_flag"] = "yes"
cont_full.loc[cont_full["is_empty_or_near_empty"], "human_review_flag"] = "yes"

aud_full.to_csv(OUTPUT_DIR / "audience_coded.csv", index=False)
cont_full.to_csv(OUTPUT_DIR / "content_coded.csv", index=False)
aud_full.to_excel(OUTPUT_DIR / "audience_coded.xlsx", index=False)
cont_full.to_excel(OUTPUT_DIR / "content_coded.xlsx", index=False)

aud_flagged = aud_full[aud_full["human_review_flag"].fillna("yes") == "yes"].copy()
cont_flagged = cont_full[cont_full["human_review_flag"].fillna("yes") == "yes"].copy()

aud_flagged.to_csv(OUTPUT_DIR / "audience_flagged_for_human_review.csv", index=False)
cont_flagged.to_csv(OUTPUT_DIR / "content_flagged_for_human_review.csv", index=False)
aud_flagged.to_excel(OUTPUT_DIR / "audience_flagged_for_human_review.xlsx", index=False)
cont_flagged.to_excel(OUTPUT_DIR / "content_flagged_for_human_review.xlsx", index=False)

log(f"Saved coded outputs. Audience flagged={len(aud_flagged)}, Content flagged={len(cont_flagged)}")
print("Saved coded files.")

## 10. Build summary tables

In [ ]:
def frequency_table(df, col):
    if col not in df.columns:
        return pd.DataFrame()
    counts = df[col].fillna("Missing").value_counts(dropna=False).rename_axis(col).reset_index(name="count")
    counts["percent"] = (counts["count"] / counts["count"].sum() * 100).round(2)
    return counts


def crosstab_count_percent(df, row, col):
    if row not in df.columns or col not in df.columns:
        return pd.DataFrame()
    ct = pd.crosstab(df[row].fillna("Missing"), df[col].fillna("Missing"))
    pct = ct.div(ct.sum(axis=1).replace(0, np.nan), axis=0).round(3) * 100
    out = ct.copy()
    out.columns = [f"count_{c}" for c in out.columns]
    for c in pct.columns:
        out[f"pct_{c}"] = pct[c]
    return out.reset_index()


def overview_table(df):
    return pd.DataFrame({
        "metric": ["total_rows", "duplicate_text_rows", "empty_or_near_empty_rows", "average_word_count", "median_word_count", "human_review_flag_yes"],
        "value": [len(df), int(df["is_duplicate_text"].sum()), int(df["is_empty_or_near_empty"].sum()), round(float(df["word_count"].mean()), 2), round(float(df["word_count"].median()), 2), int((df["human_review_flag"].fillna("yes") == "yes").sum())]
    })


def write_summary_workbook(path, sheets):
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for name, df in sheets.items():
            safe_name = name[:31].replace("/", "_").replace("\\", "_")
            if df is None or df.empty:
                df = pd.DataFrame({"note": ["No data or required columns missing"]})
            df.to_excel(writer, sheet_name=safe_name, index=False)


aud_sheets = {
    "Overview": overview_table(aud_full),
    "Relevance": frequency_table(aud_full, "relevance"),
    "Stance": frequency_table(aud_full, "stance_toward_post"),
    "Sentiment": frequency_table(aud_full, "sentiment"),
    "Emotion": frequency_table(aud_full, "emotion"),
    "Audience_Frame": frequency_table(aud_full, "audience_frame"),
    "Misogyny_Sexism": frequency_table(aud_full, "misogyny_sexism"),
    "Argument_Type": frequency_table(aud_full, "argument_type"),
    "Impact": frequency_table(aud_full, "perceived_impact"),
    "Stance_by_Influencer": crosstab_count_percent(aud_full, "std_influencer", "stance_toward_post"),
    "Emotion_by_Influencer": crosstab_count_percent(aud_full, "std_influencer", "emotion"),
    "Frame_by_Influencer": crosstab_count_percent(aud_full, "std_influencer", "audience_frame"),
    "Sexism_by_Influencer": crosstab_count_percent(aud_full, "std_influencer", "misogyny_sexism"),
    "Argument_by_Frame": crosstab_count_percent(aud_full, "audience_frame", "argument_type"),
    "Intensify_by_Frame": crosstab_count_percent(aud_full, "audience_frame", "intensifies_or_softens"),
    "Human_Review": frequency_table(aud_full, "human_review_flag"),
}

cont_sheets = {
    "Overview": overview_table(cont_full),
    "Relevance": frequency_table(cont_full, "relevance"),
    "Primary_Topic": frequency_table(cont_full, "primary_topic"),
    "Narrative": frequency_table(cont_full, "masculinity_narrative"),
    "Problem": frequency_table(cont_full, "problem_identified"),
    "Solution": frequency_table(cont_full, "solution_proposed"),
    "Main_Frame": frequency_table(cont_full, "main_frame"),
    "Orientation": frequency_table(cont_full, "content_orientation"),
    "Rhetorical_Style": frequency_table(cont_full, "rhetorical_style"),
    "Argument_Type": frequency_table(cont_full, "argument_type"),
    "Misogyny_Sexism": frequency_table(cont_full, "misogyny_sexism"),
    "Topic_by_Influencer": crosstab_count_percent(cont_full, "std_influencer", "primary_topic"),
    "Narrative_by_Influencer": crosstab_count_percent(cont_full, "std_influencer", "masculinity_narrative"),
    "Problem_by_Influencer": crosstab_count_percent(cont_full, "std_influencer", "problem_identified"),
    "Solution_by_Influencer": crosstab_count_percent(cont_full, "std_influencer", "solution_proposed"),
    "Frame_by_Influencer": crosstab_count_percent(cont_full, "std_influencer", "main_frame"),
    "Sexism_by_Influencer": crosstab_count_percent(cont_full, "std_influencer", "misogyny_sexism"),
    "Argument_by_Frame": crosstab_count_percent(cont_full, "main_frame", "argument_type"),
    "Human_Review": frequency_table(cont_full, "human_review_flag"),
}

write_summary_workbook(OUTPUT_DIR / "audience_summary_tables.xlsx", aud_sheets)
write_summary_workbook(OUTPUT_DIR / "content_summary_tables.xlsx", cont_sheets)
log("Saved summary workbooks")
print("Saved summary workbooks.")

## 11. Extract representative quotes

In [ ]:
def select_representative_quotes(df, frame_col, text_col, id_col, extra_cols, max_per_frame=5):
    if frame_col not in df.columns:
        return pd.DataFrame()
    rows = []
    work = df.copy()
    work["quote_quality_score"] = 0
    work["quote_quality_score"] += (work.get("confidence", "").fillna("") == "high").astype(int) * 3
    work["quote_quality_score"] += (work.get("relevance", "").fillna("") == "relevant").astype(int) * 2
    work["quote_quality_score"] += work.get("evidence_quote", pd.Series([""]*len(work))).fillna("").map(lambda x: 1 if len(str(x)) > 3 else 0)
    work["quote_quality_score"] += work.get("word_count", pd.Series([0]*len(work))).map(lambda x: 1 if 8 <= x <= 120 else 0)
    for frame, group in work.groupby(frame_col, dropna=False):
        group = group.sort_values("quote_quality_score", ascending=False).head(max_per_frame)
        for _, r in group.iterrows():
            item = {
                "row_id": r.get(id_col, ""),
                "frame": frame,
                "text": r.get(text_col, ""),
                "evidence_quote": r.get("evidence_quote", ""),
                "why_representative": f"High-signal example of {frame}; confidence={r.get('confidence', '')}; relevance={r.get('relevance', '')}"
            }
            for c in extra_cols:
                item[c] = r.get(c, "")
            rows.append(item)
    return pd.DataFrame(rows)


aud_quotes = select_representative_quotes(
    aud_full, "audience_frame", "comment_text_clean", "aud_row_id",
    ["std_influencer", "std_platform", "stance_toward_post", "sentiment", "emotion", "misogyny_sexism", "argument_type", "intensifies_or_softens"]
)
cont_quotes = select_representative_quotes(
    cont_full, "main_frame", "snippet_text_clean", "con_row_id",
    ["std_influencer", "std_platform", "primary_topic", "masculinity_narrative", "content_orientation", "misogyny_sexism", "argument_type", "rhetorical_style"]
)

with pd.ExcelWriter(OUTPUT_DIR / "top_representative_quotes.xlsx", engine="openpyxl") as writer:
    aud_quotes.to_excel(writer, sheet_name="Audience Quotes", index=False)
    cont_quotes.to_excel(writer, sheet_name="Content Quotes", index=False)

log(f"Saved representative quotes. Audience={len(aud_quotes)}, Content={len(cont_quotes)}")
display(aud_quotes.head())
display(cont_quotes.head())

## 12. Generate reports

In [ ]:
def md_table(df, max_rows=12):
    if df is None or df.empty:
        return "No data available."
    return df.head(max_rows).to_markdown(index=False)


def top_values_sentence(df, col, n=3):
    tab = frequency_table(df, col)
    if tab.empty:
        return "No data available."
    parts = [f"{row[col]} ({row['count']}, {row['percent']}%)" for _, row in tab.head(n).iterrows()]
    return ", ".join(parts)


def write_codebook():
    text = f"""# Codebook Used

## Audience Comment Codebook

```json
{json.dumps(AUDIENCE_CODEBOOK, indent=2)}
```

## Content Snippet Codebook

```json
{json.dumps(CONTENT_CODEBOOK, indent=2)}
```

## Notes on Interpretation

- Stance is coded toward the original post/message, not toward men or women generally.
- Misogyny/sexism is coded conservatively and should be reviewed by humans for high-stakes labels.
- Frames identify the dominant interpretive frame in the row, not every possible theme.
- Confidence reflects coding clarity, not whether the statement is true.
"""
    (OUTPUT_DIR / "codebook_used.md").write_text(text, encoding="utf-8")


def write_methodology():
    text = f"""# Methodology Note

## Files Analyzed

- Audience comments: `{AUDIENCE_FILE}`
- Content snippets: `{CONTENT_FILE}`

## Row Counts

- Audience comments analyzed: {len(aud_full)}
- Content snippets analyzed: {len(cont_full)}

## Coding Approach

This notebook used the OpenAI API with structured JSON output to code rows using fixed closed-ended codebooks. Rows were coded in batches of {BATCH_SIZE}. Each coded row includes labels, confidence, human review flag, an evidence quote, and a brief coding rationale.

## Cleaning

Text was normalized for whitespace and preserved in original form. Empty/near-empty rows and duplicate text rows were flagged.

## Human Review

Rows were flagged for human review when the LLM marked them as ambiguous, low-confidence, mixed, hostile/dehumanizing, or otherwise potentially difficult to interpret. Human validation is recommended before making final claims.

## Limitations

- The dataset is a high-signal selected sample and should not be treated as nationally representative.
- LLM labels are interpretive and should be validated with human coding.
- Sarcasm, slang, multilingual comments, and context-dependent references can be misclassified.
- Platform differences may affect audience behavior and should not be interpreted as purely ideological differences.
- Influencer-level and frame-level claims are safer than broad country-level claims.
"""
    (OUTPUT_DIR / "methodology_note.md").write_text(text, encoding="utf-8")


def write_combined_report():
    report = f"""# LLM-Assisted Exploratory Data Analysis Report

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}

## 1. Executive Summary

- The analysis coded **{len(aud_full)} audience comments** and **{len(cont_full)} content snippets** using fixed categories for stance, framing, sentiment, emotion, argument type, and misogyny/sexism.
- Audience comments most often used these frames: **{top_values_sentence(aud_full, 'audience_frame')}**.
- Content snippets most often used these frames: **{top_values_sentence(cont_full, 'main_frame')}**.
- Audience stance distribution: **{top_values_sentence(aud_full, 'stance_toward_post')}**.
- Content orientation distribution: **{top_values_sentence(cont_full, 'content_orientation')}**.
- Audience misogyny/sexism distribution: **{top_values_sentence(aud_full, 'misogyny_sexism')}**.
- Content misogyny/sexism distribution: **{top_values_sentence(cont_full, 'misogyny_sexism')}**.
- Human review is recommended for **{int((aud_full['human_review_flag'].fillna('yes') == 'yes').sum())} audience rows** and **{int((cont_full['human_review_flag'].fillna('yes') == 'yes').sum())} content rows**.

## 2. Dataset Overview

### Audience Overview

{md_table(overview_table(aud_full))}

### Content Overview

{md_table(overview_table(cont_full))}

## 3. Audience Reception Findings

### Stance Toward Original Post

{md_table(frequency_table(aud_full, 'stance_toward_post'))}

### Audience Frames

{md_table(frequency_table(aud_full, 'audience_frame'))}

### Emotion

{md_table(frequency_table(aud_full, 'emotion'))}

### Misogyny / Sexism in Audience Comments

{md_table(frequency_table(aud_full, 'misogyny_sexism'))}

### Argument Types in Audience Comments

{md_table(frequency_table(aud_full, 'argument_type'))}

## 4. Content Analysis Findings

### Primary Topics

{md_table(frequency_table(cont_full, 'primary_topic'))}

### Masculinity Narratives

{md_table(frequency_table(cont_full, 'masculinity_narrative'))}

### Main Content Frames

{md_table(frequency_table(cont_full, 'main_frame'))}

### Content Orientation

{md_table(frequency_table(cont_full, 'content_orientation'))}

### Problems Identified

{md_table(frequency_table(cont_full, 'problem_identified'))}

### Solutions Proposed

{md_table(frequency_table(cont_full, 'solution_proposed'))}

### Rhetorical Styles

{md_table(frequency_table(cont_full, 'rhetorical_style'))}

## 5. Linked Content–Audience Insights

Use the summary workbooks to connect creator messages to audience uptake. The strongest linked insights should compare:

- which content frames receive supportive versus oppositional comments;
- which audience comments intensify, soften, extend, or challenge the original message;
- which themes are repeated by audiences rather than simply liked;
- where comments shift from the creator message into broader gender-war discourse;
- where audience members use personal testimony, trauma disclosure, faith, or advice to interpret the content.

Safe reporting formulation: **In this selected high-signal sample, audience comments often took up creator frames by repeating, extending, challenging, or softening them. These patterns should be interpreted as audience reception within the sampled content, not as country-level public opinion.**

## 6. Framing Analysis

Major frames to examine across both datasets include:

- female blame / modern women as threat
- male victimhood / men as exploited or ignored
- provider pressure / status anxiety
- traditional gender order / submission and hierarchy
- trauma healing / father wounds / emotional openness
- faith, morality, and testimony
- social accountability / anti-violence
- fatherhood and responsibility

Use `top_representative_quotes.xlsx` for quote-backed slide evidence.

## 7. Argument Mining Findings

Audience argument types:

{md_table(frequency_table(aud_full, 'argument_type'))}

Content argument types:

{md_table(frequency_table(cont_full, 'argument_type'))}

Interpretive guide: regressive discourse may rely more on broad gender generalizations, common-sense claims, insult/mockery, or advice rules; progressive/vulnerability-forward discourse may rely more on testimony, personal experience, faith, trauma disclosure, or social accountability. Confirm this using the crosstabs in the summary workbooks before making final claims.

## 8. Human Review Priorities

Prioritize reviewing low-confidence rows, mixed/unclear stance rows, hostile/dehumanizing sexism labels, slang-heavy or sarcastic rows, weak evidence quotes, and complex snippets with multiple claims.

## 9. Recommended Slide Tables

1. Audience stance distribution.
2. Audience frame by influencer.
3. Emotion by audience frame.
4. Misogyny/sexism level by influencer.
5. Content topic by influencer.
6. Masculinity narrative by influencer.
7. Content orientation by influencer.
8. Argument type by frame.
9. Representative quote slides for each major frame.
10. Content–audience uptake matrix: original frame → audience stance → intensification/softening.

## 10. Limitations

- The sample is selected and high-signal, not representative of Kenya, Nigeria, or all young men.
- LLM coding should be validated by human coders before final reporting.
- Short comments, sarcasm, multilingual language, and platform-specific slang may reduce accuracy.
- Country-level claims should be avoided unless the sampling design supports them.
- Safer claims compare frames, narrative modes, rhetorical strategies, and audience uptake within this dataset.

## 11. Appendix: Key Tables

### Audience Frame by Influencer

{md_table(crosstab_count_percent(aud_full, 'std_influencer', 'audience_frame'), max_rows=20)}

### Content Frame by Influencer

{md_table(crosstab_count_percent(cont_full, 'std_influencer', 'main_frame'), max_rows=20)}
"""
    (OUTPUT_DIR / "combined_insights_report.md").write_text(report, encoding="utf-8")

write_codebook()
write_methodology()
write_combined_report()
log("Saved codebook, methodology note, and combined report")
print("Reports saved.")

## 13. Optional quick charts

In [ ]:
import matplotlib.pyplot as plt
chart_dir = OUTPUT_DIR / "charts"
chart_dir.mkdir(exist_ok=True)

def save_bar_chart(freq_df, label_col, title, filename, top_n=10):
    if freq_df.empty or label_col not in freq_df.columns:
        return
    data = freq_df.head(top_n).iloc[::-1]
    plt.figure(figsize=(8, 5))
    plt.barh(data[label_col].astype(str), data["count"])
    plt.title(title)
    plt.xlabel("Count")
    plt.tight_layout()
    plt.savefig(chart_dir / filename, dpi=200)
    plt.close()

save_bar_chart(frequency_table(aud_full, "audience_frame"), "audience_frame", "Audience Frames", "audience_frames.png")
save_bar_chart(frequency_table(aud_full, "stance_toward_post"), "stance_toward_post", "Audience Stance Toward Post", "audience_stance.png")
save_bar_chart(frequency_table(cont_full, "main_frame"), "main_frame", "Content Main Frames", "content_frames.png")
save_bar_chart(frequency_table(cont_full, "content_orientation"), "content_orientation", "Content Orientation", "content_orientation.png")
log("Saved optional charts")
print(f"Charts saved to {chart_dir}")

## 14. Final output check

In [ ]:
expected = [
    "audience_cleaned.csv", "content_cleaned.csv", "audience_coded.csv", "content_coded.csv",
    "audience_coded.xlsx", "content_coded.xlsx", "audience_flagged_for_human_review.csv",
    "content_flagged_for_human_review.csv", "audience_summary_tables.xlsx", "content_summary_tables.xlsx",
    "combined_insights_report.md", "codebook_used.md", "methodology_note.md", "top_representative_quotes.xlsx", "analysis_log.txt"
]
print("Output files:")
for f in expected:
    path = OUTPUT_DIR / f
    print(f"{'✓' if path.exists() else '✗'} {path}")
print("\nAnalysis complete. Outputs saved in outputs_llm_eda/")